In [6]:
# 1) Import necessary libraries
"""
Rhea Soil Nutrient Prediction Challenge
Uses ExtraTreesRegressor with a two-stage approach:
  Stage 1: Predict abundant nutrients (Al, Ca, Cu, Fe, K, Mg, Mn, N) from geo + depth + time features
  Stage 2: Predict sparse nutrients (B, Na, P, S, Zn) using Stage 1 features + Stage 1 predictions
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

In [7]:
# 2)Load data
train    = pd.read_csv('/content/Train.csv')
test     = pd.read_csv('/content/TestSet.csv')
keep     = pd.read_csv('/content/TargetPred_To_Keep.csv')
dates    = pd.read_csv('/content/Sample_Collection_Dates.csv')
sub_fmt  = pd.read_csv('/content/SampleSubmission.csv')

print(f"Train: {train.shape}, Test: {test.shape}")

Train: (44298, 25), Test: (6070, 17)


In [8]:
# 3) DATE FEATURES (from Sample_Collection_Dates.csv)
def parse_date_features(df_dates, id_col='ID'):
    d = df_dates.copy()
    d['start_dt'] = pd.to_datetime(d['start_date'], dayfirst=True, errors='coerce')
    d['end_dt']   = pd.to_datetime(d['end_date'],   dayfirst=True, errors='coerce')
    d['start_year']  = d['start_dt'].dt.year
    d['start_month'] = d['start_dt'].dt.month
    d['end_year']    = d['end_dt'].dt.year
    d['date_range_days'] = (d['end_dt'] - d['start_dt']).dt.days
    return d[[id_col, 'start_year', 'start_month', 'end_year', 'date_range_days']]

date_feats = parse_date_features(dates)

In [9]:
# 4) FEATURE ENGINEERING
def build_features(df, date_feats):
    """Create feature matrix from a dataframe."""
    d = df.copy()

    # Depth as binary
    d['depth_is_deep'] = (d['Depth_cm'] == '20-50').astype(int)

    # Merge date features
    d = d.merge(date_feats, on='ID', how='left')

    # Geo-derived features
    d['lat_abs']      = d['Latitude'].abs()
    d['lon_abs']      = d['Longitude'].abs()
    d['lat_lon_prod'] = d['Latitude'] * d['Longitude']
    d['lat_sq']       = d['Latitude'] ** 2
    d['lon_sq']       = d['Longitude'] ** 2

    # Distance from equator (0°) and prime meridian (0°)
    d['dist_equator'] = d['lat_abs']

    # Approximate distance from centroid of Kenya (approximate center of dataset)
    CENTER_LAT, CENTER_LON = -1.5, 37.5
    d['dist_center'] = np.sqrt((d['Latitude'] - CENTER_LAT)**2 + (d['Longitude'] - CENTER_LON)**2)

    # Quadrant encoding
    d['lat_pos'] = (d['Latitude'] > 0).astype(int)
    d['lon_pos'] = (d['Longitude'] > 0).astype(int)

    base_cols = ['Latitude', 'Longitude', 'depth_is_deep',
                 'lat_abs', 'lon_abs', 'lat_lon_prod', 'lat_sq', 'lon_sq',
                 'dist_equator', 'dist_center', 'lat_pos', 'lon_pos',
                 'start_year', 'start_month', 'end_year', 'date_range_days']

    # Fill date NaN with median
    for c in ['start_year', 'start_month', 'end_year', 'date_range_days']:
        d[c] = d[c].fillna(d[c].median())

    return d, base_cols

train, base_cols = build_features(train, date_feats)
test,  _         = build_features(test, date_feats)

print(f"Base features: {len(base_cols)}")
print(f"Base feature cols: {base_cols}")

Base features: 16
Base feature cols: ['Latitude', 'Longitude', 'depth_is_deep', 'lat_abs', 'lon_abs', 'lat_lon_prod', 'lat_sq', 'lon_sq', 'dist_equator', 'dist_center', 'lat_pos', 'lon_pos', 'start_year', 'start_month', 'end_year', 'date_range_days']


In [10]:
# 5) DEFINE TARGETS
# Stage 1: abundant targets (>99% coverage in train)
stage1_targets = ['Al', 'Ca', 'Cu', 'Fe', 'K', 'Mg', 'Mn', 'N']

# Stage 2: sparse targets (~4% coverage in train) – use stage1 preds as extra features
stage2_targets = ['B', 'Na', 'P', 'S', 'Zn']

all_targets = stage1_targets + stage2_targets

In [11]:
# 6) MODEL CONFIG
MODEL_PARAMS = dict(
    n_estimators=500,
    max_features='sqrt',
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)

In [12]:
# 7a) TRAIN & PREDICT ABUNDANT NUTRIENTS
stage1_train_preds = {}  # OOB-style, used as features for stage2 training
stage1_test_preds  = {}

for tgt in stage1_targets:
    mask = train[tgt].notna()
    X_tr = train.loc[mask, base_cols].values
    y_tr = train.loc[mask, tgt].values
    X_te = test[base_cols].values

    print(f"  {tgt}: {mask.sum()} train samples → training ExtraTrees...")

    model = ExtraTreesRegressor(**MODEL_PARAMS)
    model.fit(X_tr, y_tr)

    # Predict on test
    stage1_test_preds[tgt] = model.predict(X_te)

    # Predict on ALL train rows (needed as features for stage2 on overlapping rows)
    stage1_train_preds[tgt] = model.predict(train[base_cols].values)

    print(f"    Test pred range: [{stage1_test_preds[tgt].min():.2f}, {stage1_test_preds[tgt].max():.2f}]")

  Al: 44296 train samples → training ExtraTrees...
    Test pred range: [155.33, 1977.55]
  Ca: 44298 train samples → training ExtraTrees...
    Test pred range: [136.74, 22034.33]
  Cu: 44257 train samples → training ExtraTrees...
    Test pred range: [0.11, 10.11]
  Fe: 44257 train samples → training ExtraTrees...
    Test pred range: [30.48, 242.73]
  K: 44298 train samples → training ExtraTrees...
    Test pred range: [18.91, 1807.01]
  Mg: 44298 train samples → training ExtraTrees...
    Test pred range: [37.03, 2905.40]
  Mn: 44255 train samples → training ExtraTrees...
    Test pred range: [9.48, 376.83]
  N: 44257 train samples → training ExtraTrees...
    Test pred range: [0.18, 6.23]


In [13]:
# 7b) TRAIN & PREDICT SPARSE NUTRIENTS
# Add stage1 predicted values as features
stage1_pred_cols = [f'pred_{t}' for t in stage1_targets]
for t in stage1_targets:
    train[f'pred_{t}'] = stage1_train_preds[t]
    test[f'pred_{t}']  = stage1_test_preds[t]

# Also add C_organic and ph as features for stage2 training (available in train)
# For test we use stage1 predictions as proxies, not C_organic/ph (not in test)
# So we only add them where available in train

extended_cols = base_cols + stage1_pred_cols

# Add C_organic and ph for stage2 training only (not available in test)
train_stage2_cols = extended_cols + ['C_organic', 'ph']

# Fill C_organic and ph NaN with median for training rows
train['C_organic'] = train['C_organic'].fillna(train['C_organic'].median())
train['ph']        = train['ph'].fillna(train['ph'].median())

stage2_test_preds = {}

for tgt in stage2_targets:
    mask = train[tgt].notna()

    # Try using richer features (C_organic, ph) for stage2 models
    # We use only extended_cols (no C_organic/ph) for test prediction
    X_tr = train.loc[mask, train_stage2_cols].values
    y_tr = train.loc[mask, tgt].values
    X_te = test[extended_cols].values

    # Build a version of test that matches train_stage2_cols:
    # fill C_organic and ph with their train medians
    C_org_median = train['C_organic'].median()
    ph_median    = train['ph'].median()
    test_stage2  = test[extended_cols].copy()
    test_stage2['C_organic'] = C_org_median
    test_stage2['ph']        = ph_median
    X_te_full = test_stage2[extended_cols + ['C_organic', 'ph']].values

    print(f"  {tgt}: {mask.sum()} train samples → training ExtraTrees...")

    model = ExtraTreesRegressor(**MODEL_PARAMS)
    model.fit(X_tr, y_tr)

    preds = model.predict(X_te_full)
    preds = np.clip(preds, 0, None)   # nutrients can't be negative
    stage2_test_preds[tgt] = preds

    print(f"    Test pred range: [{preds.min():.2f}, {preds.max():.2f}]")


  B: 1909 train samples → training ExtraTrees...
    Test pred range: [0.01, 0.34]
  Na: 1948 train samples → training ExtraTrees...
    Test pred range: [13.06, 75.55]
  P: 1909 train samples → training ExtraTrees...
    Test pred range: [2.17, 57.12]
  S: 1909 train samples → training ExtraTrees...
    Test pred range: [4.73, 19.94]
  Zn: 1909 train samples → training ExtraTrees...
    Test pred range: [0.71, 1.88]


In [14]:
# 8) ASSEMBLE SUBMISSION
sub = sub_fmt.copy()

target_map = {
    'Al': 'Target_Al', 'B': 'Target_B', 'Ca': 'Target_Ca',
    'Cu': 'Target_Cu', 'Fe': 'Target_Fe', 'K': 'Target_K',
    'Mg': 'Target_Mg', 'Mn': 'Target_Mn', 'N': 'Target_N',
    'Na': 'Target_Na', 'P': 'Target_P', 'S': 'Target_S',
    'Zn': 'Target_Zn'
}

# Build a lookup from test ID to row index
test_id_to_idx = {row['ID']: i for i, row in test.iterrows()}

# Fill predictions into submission
all_preds = {**{t: stage1_test_preds[t] for t in stage1_targets},
             **{t: stage2_test_preds[t]  for t in stage2_targets}}

for tgt, sub_col in target_map.items():
    preds_arr = all_preds[tgt]
    # Map test rows in order of sub_fmt IDs
    pred_series = pd.Series(preds_arr, index=test.index)
    # test and sub should be aligned by ID
    test_indexed = test[['ID']].copy()
    test_indexed['pred'] = preds_arr
    merged = sub[['ID']].merge(test_indexed, on='ID', how='left')
    sub[sub_col] = merged['pred'].values


In [15]:
# 9) APPLY TargetPred_To_Keep MASK (zero out entries where keep==0)
keep_indexed = keep.set_index('ID')
sub_indexed  = sub.set_index('ID')

for tgt, sub_col in target_map.items():
    if tgt in keep_indexed.columns:
        mask_series = keep_indexed[tgt]
        # Where keep == 0, zero out prediction
        sub_indexed.loc[sub_indexed.index, sub_col] = np.where(
            mask_series.reindex(sub_indexed.index, fill_value=0) == 0,
            0.0,
            sub_indexed[sub_col]
        )

sub = sub_indexed.reset_index()

# Clip negatives to 0
for sub_col in target_map.values():
    sub[sub_col] = sub[sub_col].clip(lower=0)

In [18]:
import os

# 10) SAVE
out_path = '/content/submissions/submissionv1.csv'

# Create the output directory if it doesn't exist
os.makedirs(os.path.dirname(out_path), exist_ok=True)

sub.to_csv(out_path, index=False)
print(f"\nSubmission saved → {out_path}")
print(f"Shape: {sub.shape}")
print(sub.head())
print("\nNon-zero counts per target:")
for col in target_map.values():
    print(f"  {col}: {(sub[col] > 0).sum()}")


Submission saved → /content/submissions/submissionv1.csv
Shape: (6070, 14)
       ID    Target_Al  Target_B    Target_Ca  Target_Cu   Target_Fe  \
0  00A83S   837.325932  0.293620  3553.408185   3.863637  114.731796   
1  00F2Q3   765.478895  0.272776  3714.480378   2.897161  131.797799   
2  00FNFP   613.634226  0.000000  1444.721682   1.812782  107.806836   
3  01MFSS  1241.719757  0.000000  1124.886410   2.305965  108.240434   
4  02851F  1175.270914  0.000000  2995.986164   4.380949  108.753041   

     Target_K   Target_Mg   Target_Mn  Target_N  Target_Na   Target_P  \
0  680.936981  644.450730  167.064758  2.167535  53.468722  29.992611   
1  436.910020  502.797796  164.596429  1.127819  55.048090  29.494562   
2  219.796191  324.537161   99.999310  0.512897   0.000000   0.000000   
3  244.228429  297.943961  126.823033  2.376107   0.000000   0.000000   
4  310.164620  461.464013  226.962771  2.585538   0.000000   0.000000   

    Target_S  Target_Zn  
0  13.900646   1.614659  
